 Main Orchestrator Agent
    ├── call_essay_writer  → Essay Writer Sub-agent (uses web_search)
    └── call_image_suggester → Image Suggester Sub-agent (uses web_search)


In [1]:
from dotenv import load_dotenv
from langchain.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from tavily import TavilyClient
from typing import Dict, Any

load_dotenv()

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

# verify imports worked
print("All imports OK")


All imports OK


In [2]:
m1=ChatAnthropic(model="claude-haiku-4-5-20251001")

In [3]:
# Sub-agents + their tool wrappers — all in one cell
essay_writer = create_agent(
    model=ChatAnthropic(model="claude-haiku-4-5-20251001"),
    tools=[web_search],
    system_prompt="""You are an expert essay writer.
When given a topic, search the web for accurate information,
then write a well-structured essay with an introduction, body paragraphs, and conclusion."""
)

image_suggester = create_agent(
    model=ChatAnthropic(model="claude-haiku-4-5-20251001"),
    tools=[web_search],
    system_prompt="""You are a visual content curator.
When given a topic, search the web and suggest 5 relevant images with:
- A descriptive title
- What the image shows
- Why it is relevant to the topic
Format your response as a numbered list."""
)

@tool
def call_essay_writer(topic: str) -> str:
    """Call the essay writer agent to research and write an essay about the given topic"""
    response = essay_writer.invoke(
        {"messages": [HumanMessage(content=f"Write a detailed essay about: {topic}")]}
    )
    content = response["messages"][-1].content
    if isinstance(content, list):
        return "".join(block["text"] for block in content if block.get("type") == "text")
    return content

@tool
def call_image_suggester(topic: str) -> str:
    """Call the image suggester agent to find relevant images for the given topic"""
    response = image_suggester.invoke(
        {"messages": [HumanMessage(content=f"Suggest relevant images for the topic: {topic}")]}
    )
    content = response["messages"][-1].content
    if isinstance(content, list):
        return "".join(block["text"] for block in content if block.get("type") == "text")
    return content

print("Sub-agents and tools ready")


Sub-agents and tools ready


In [4]:
main_agent = create_agent(
    model=m1,
    tools=[web_search],
    system_prompt="""You are an expert academic essay writer specializing in international law and geopolitical analysis.

Your task is to write a complete, well-structured essay in proper prose — NOT bullet points, NOT headers, NOT a list of facts.

The essay MUST follow this structure:

INTRODUCTION (2-3 paragraphs):
- Open with a compelling statement about Palestine and its people
- Provide brief historical background on Palestinian land before the occupation
- State the thesis: Israel is an occupying power in violation of international law

BODY (5-6 paragraphs, each focused on one argument):
- Paragraph 1: Historical displacement — the Nakba of 1948 and the forced expulsion of Palestinians
- Paragraph 2: Israel as an occupying power under international law — cite UN Resolutions 242, 338, 2334 and the Fourth Geneva Convention
- Paragraph 3: The ICJ rulings — the 2004 advisory opinion on the separation wall and the 2024 ruling declaring the occupation unlawful
- Paragraph 4: Ongoing violations — illegal settlements, the Gaza blockade, collective punishment of civilians
- Paragraph 5: The weaponization of religion — how religious narratives are used to justify occupation despite international law not recognizing religious claims to territory
- Paragraph 6: The humanitarian cost — casualties, displacement, destruction in Gaza and the West Bank with real figures

CONCLUSION (1-2 paragraphs):
- Summarize the key arguments
- Call for accountability under international law and justice for the Palestinian people

Write in formal academic prose. Each paragraph must flow naturally into the next. Use transitional phrases. Do NOT use bullet points, numbered lists, or section headers in the essay body. Search the web for accurate facts, figures, and legal citations before writing."""
)


In [5]:
config = {"configurable": {"thread_id": "1"}}

response = main_agent.invoke(
{"messages": [HumanMessage(content="Write a formal academic essay about Palestine and the Israeli occupation. The essay must have a proper introduction, body paragraphs, and conclusion in flowing prose. No bullet points or headers.")]},    
config=config
)

content = response["messages"][-1].content
if isinstance(content, list):
    full_text = "".join(block["text"] for block in content if block.get("type") == "text")
else:
    full_text = content

print(full_text)


Now I have sufficient information to write a comprehensive academic essay. Let me compose it in proper prose format with flowing paragraphs.

---

# ISRAEL'S OCCUPATION OF PALESTINE: A VIOLATION OF INTERNATIONAL LAW AND HUMAN RIGHTS

The Palestinian people, numbering over 5 million worldwide today, represent one of the world's most persistent cases of statelessness and displacement in the modern era. For more than seven decades, Palestinians have endured systematic dispossession from their ancestral lands, fragmentation of their territory, and denial of fundamental self-determination rights that international law explicitly recognizes as fundamental to all peoples. The occupation of Palestinian territories by Israel, which has persisted since 1967 and continues in various forms today, constitutes a clear and documented violation of international law, contravening established principles of the United Nations Charter, the Fourth Geneva Convention, and multiple binding Security Council re

In [6]:
from docx import Document
from docx.shared import Pt
import re

def add_formatted_paragraph(doc, text):
    p = doc.add_paragraph()
    # split on bold (**text**) and italic (*text*)
    parts = re.split(r'(\*\*.*?\*\*|\*.*?\*)', text)
    for part in parts:
        if part.startswith('**') and part.endswith('**'):
            run = p.add_run(part[2:-2])
            run.bold = True
        elif part.startswith('*') and part.endswith('*'):
            run = p.add_run(part[1:-1])
            run.italic = True
        else:
            p.add_run(part)
    return p

doc = Document()

for line in full_text.split("\n"):
    line = line.strip()
    if not line:
        continue
    elif line.startswith("# "):
        doc.add_heading(line[2:].strip(), level=1)
    elif line.startswith("## "):
        doc.add_heading(line[3:].strip(), level=2)
    elif line.startswith("### "):
        doc.add_heading(line[4:].strip(), level=3)
    elif re.match(r'^[-*]\s', line):               # bullet point
        p = add_formatted_paragraph(doc, line[2:])
        p.style = 'List Bullet'
    elif re.match(r'^\d+\.\s', line):              # numbered list
        p = add_formatted_paragraph(doc, re.sub(r'^\d+\.\s', '', line))
        p.style = 'List Number'
    else:
        add_formatted_paragraph(doc, line)

doc.save("Palestine_Essay.docx")
print("Saved as Palestine_Essay.docx")


Saved as Palestine_Essay.docx
